# Financial Reducto OracleDB Demo Notebook

This notebook demonstrates the financial version of the Reducto + Oracle flow:

1. Load `.env`.
2. Connect to Oracle.
3. Inspect stored documents.
4. Ask a polished question with evidence.
5. Optionally ingest a new document URL.


## Platform Context

![Executive thesis: Reducto AI + Oracle Database](assets/reducto_oracle_executive_thesis.svg)

![Capability map: Oracle Database approach vs standalone vector platforms](assets/reducto_oracle_platform_capability_map.svg)

![Operating model and risk controls](assets/reducto_oracle_operating_model_risk_controls.svg)


In [1]:
import os
import sys
from pathlib import Path


def find_sdk_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "src" / "reducto" / "lib" / "oracledb").exists():
            return path
    raise RuntimeError("Could not locate the reducto-python-sdk repository root")


ROOT = find_sdk_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_env(path=ROOT / "examples" / "oracledb" / ".env"):
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("'\""))


load_env()
print("Oracle user:", os.getenv("ORACLE_USER"))
print("Oracle DSN:", os.getenv("ORACLE_DSN"))
print("Reducto key set:", bool(os.getenv("REDUCTO_API_KEY")))
print("SEC user-agent set:", bool(os.getenv("SEC_USER_AGENT")))

Oracle user: REDUCTO_RAG_APP
Oracle DSN: localhost:1521/FREEPDB1
Reducto key set: True
SEC user-agent set: True


## Connect to Oracle and inspect the schema


In [2]:
from reducto.lib.oracledb.config import connect_oracle, vector_dimensions_from_env
from reducto.lib.oracledb.oracle import OracleSchemaManager

connection = connect_oracle()
OracleSchemaManager(connection).create_schema(vector_dimensions=vector_dimensions_from_env())
print("Oracle schema is ready")

for table in ["DOCUMENTS", "DOCUMENT_CHUNKS", "PARSED_TABLES", "FINANCIAL_FACTS"]:
    with connection.cursor() as cursor:
        cursor.execute(f"select count(*) from {table}")
        print(table, cursor.fetchone()[0])

Oracle schema is ready
DOCUMENTS 1
DOCUMENT_CHUNKS 1
PARSED_TABLES 47
FINANCIAL_FACTS 1222


## Ingest a new URL

Uncomment and run this cell to parse and store another SEC filing. This calls Reducto and may take a while.


In [ ]:
from reducto.lib.oracledb.models import DocumentMetadata
from reducto.lib.oracledb.oracle import OracleDocumentRepository
from reducto.lib.oracledb.reducto_client import ReductoDocumentParser

url = "https://www.sec.gov/Archives/edgar/data/1045810/000104581024000029/nvda-20240128.htm"
parse_result = ReductoDocumentParser().parse_url(url)
metadata = DocumentMetadata(
    source_uri=url,
    source_kind="url",
    company="NVDA",
    fiscal_year=2024,
    filing_type="10-K",
    title="NVIDIA 2024 Form 10-K",
)
document_id = OracleDocumentRepository(connection).store_parse_result(
    metadata,
    parse_result,
    embedding_provider_from_env(input_type="search_document"),
)
print("Stored document", document_id)

## List stored documents


In [3]:
with connection.cursor() as cursor:
    cursor.execute("""
        select document_id, company, fiscal_year, filing_type, title, source_uri
        from documents
        order by created_at desc
        fetch first 10 rows only
    """)
    rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'AAPL', 2023, '10-K', 'Apple 2023 Form 10-K', 'https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm')


## Ask a polished question

This uses Oracle vector search plus the extractive Q&A formatter in `qa.py`.


In [4]:
from reducto.lib.oracledb.qa import format_answer, answer_from_search_results
from reducto.lib.oracledb.models import SearchFilters
from reducto.lib.oracledb.retrieval import OracleHybridRetriever
from reducto.lib.oracledb.embeddings import embedding_provider_from_env

question = "What were Apple's net sales in 2023?"
filters = SearchFilters(company="AAPL", fiscal_year=2023)
retriever = OracleHybridRetriever(
    connection,
    embedding_provider_from_env(input_type="search_query"),
)
results = retriever.semantic_search(question, filters=filters, limit=5)
answer = answer_from_search_results(question, results, evidence_limit=3)
print(format_answer(answer))

Question
  What were Apple's net sales in 2023?

Answer
  The Company's total net sales were $383.3 billion and net income was $97.0 billion during 2023.

Evidence
  1. page 54
     Fiscal Year Highlights The Company's total net sales were $383.3 billion and net income was $97.0 billion during 2023. The Company's total net sales decreased 3% or $11.0 billion during 2023 compared to 2022. The weakness in foreign currencies relative to the U.S. dollar accounted for more than the entire year-over-year decrease in total net sales, which consisted primarily of lower net sales of Mac and iPhone, partially offset by higher net sales of Services. The Company announces new product, service and software offerings at various times during the year. Significant announcements during fiscal year 2023 included the following: First Quarter 2023: •iPad and iPad Pro; •Next-generation Apple TV 4K; and •MLS Season Pass, a Major League Soccer subscription streaming service. Second Quarter 2023: •MacBook Pro

In [ ]:
connection.close()